# Dellmology Pro - CNN Pattern Recognition Training

Jalankan notebook ini di Google Colab. Pastikan Anda telah mengunggah folder `dataset/` (yang berisi gambar-gambar yang sudah dilabeli dalam folder seperti `bullish_flag`, `sideways`, `breakout`) ke Google Drive Anda.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
import os

# Ganti dengan path ke dataset Anda di Google Drive
data_dir = '/content/drive/MyDrive/dellmology_dataset'
batch_size = 32
num_epochs = 15
num_classes = 3 # Ubah sesuai jumlah folder kategori Anda

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# Jika Anda memisahkan folder train dan val
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x),
                                          data_transforms[x])
                  for x in ['train', 'val']}
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=batch_size,
                                             shuffle=True, num_workers=4)
              for x in ['train', 'val']}

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Training on {device}")

In [ ]:
model = models.resnet18(pretrained=True)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

for epoch in range(num_epochs):
    print(f'Epoch {epoch}/{num_epochs - 1}')
    print('-' * 10)

    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
        else:
            model.eval()

        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in dataloaders[phase]:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_loss = running_loss / len(image_datasets[phase])
        epoch_acc = running_corrects.double() / len(image_datasets[phase])

        print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

print("Training complete.")

In [ ]:
# Export the trained model
torch.save(model.state_dict(), '/content/drive/MyDrive/dellmology_dataset/pattern_model.pt')
print("Model saved to pattern_model.pt. Please download it and place it in the cnn-worker folder.")
# Save class names
class_names = image_datasets['train'].classes
with open('/content/drive/MyDrive/dellmology_dataset/classes.txt', 'w') as f:
    for c in class_names:
        f.write(c + '\n')
print("Classes saved.")